# Teacher B — SeamlessM4T v2

## Copy/run order

1. Block 1 — configuration
2. Block 2 — Phase 0A runtime probe
3. Block 3 — Phase 0B bundle and audio verification
4. Block 4 — stop gate
5. Block 5 — dependencies (after approval)
6. Block 6 — shared utilities
7. Block 7 — immutable model revision
8. Block 8 — SeamlessM4T generation
9. Block 9 — Hindi-2K tokenizer audit
10. Block 10 — metadata
11. Block 11 — Teacher B metrics, diagnostics, and manual examples
12. Block 12 — persistent artifact archive

This notebook independently benchmarks English speech → Hindi text using `facebook/seamless-m4t-v2-large`. Outputs are written incrementally to Google Drive so they survive Colab termination. It does not generate Hindi speech, use Nadantara's vocoder, or modify any source manifest.

Run the Phase 0 runtime/file cells first and stop for approval. The model cell is deliberately later. Official API: [SeamlessM4T v2](https://huggingface.co/facebook/seamless-m4t-v2-large) and [Transformers documentation](https://huggingface.co/docs/transformers/main/en/model_doc/seamless_m4t_v2).

In [ ]:
# BLOCK 1 — configuration and benchmark bundle paths.
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
from pathlib import Path
import platform
import statistics
import tarfile
import time
import unicodedata

import numpy as np
import torch

MODEL_ID = 'facebook/seamless-m4t-v2-large'
from google.colab import drive
drive.mount('/content/drive')
PERSIST_ROOT = Path('/content/drive/MyDrive/Nadantara/teacher_b_100')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE_ARCHIVE = Path('/content/wordproject_validation_100_seed_20260918.tar.gz')
BUNDLE_ROOT = Path('/content/wordproject_validation_100_seed_20260918')
OUTPUT_ROOT = PERSIST_ROOT / 'outputs'
EXPECTED_BENCHMARK_EXAMPLES = 100
PERSIST_BENCHMARK_ROOT = PERSIST_ROOT / 'benchmark'
RETRY_FAILED = False
MAX_AUDIO_SECONDS = 30.0
MAX_TRANSLATION_TOKENS = 256
print('python=', platform.python_version())
print('model=', MODEL_ID)
print('persistent_root=', PERSIST_ROOT)

In [ ]:
# BLOCK 2 — PHASE 0A runtime probe; this cell downloads no models.
import shutil
print('python=', platform.python_version())
print('torch=', torch.__version__)
print('torch_cuda_version=', torch.version.cuda)
print('cuda_available=', torch.cuda.is_available())
print('content_exists=', Path('/content').is_dir())
print('free_disk_gib=', round(shutil.disk_usage('/content').free / 1024**3, 3))
if not torch.cuda.is_available():
    raise RuntimeError('PHASE 0 STOP: no CUDA GPU is available')
device_index = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device_index)
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
print('gpu_index=', device_index)
print('gpu_name=', props.name)
print('gpu_vram_gib=', round(props.total_memory / 1024**3, 3))
print('gpu_free_memory_gib=', round(free_bytes / 1024**3, 3))
print('gpu_total_memory_gib=', round(total_bytes / 1024**3, 3))
print('PHASE_0_RUNTIME_PASS')

In [ ]:
# BLOCK 3 — PHASE 0B uploaded-file access, manifest identity, and audio-byte checks.
if not BUNDLE_ARCHIVE.is_file():
    raise FileNotFoundError(f'upload the benchmark archive first: {BUNDLE_ARCHIVE}')
def safe_extract(archive: Path, destination: Path) -> None:
    with tarfile.open(archive, 'r:gz') as tar:
        root = destination.parent.resolve()
        for member in tar.getmembers():
            target = (destination.parent / member.name).resolve()
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        tar.extractall(destination.parent)
if not (BUNDLE_ROOT / 'benchmark_manifest.jsonl').is_file():
    safe_extract(BUNDLE_ARCHIVE, BUNDLE_ROOT)
manifest_path = BUNDLE_ROOT / 'benchmark_manifest.jsonl'
metadata_path = BUNDLE_ROOT / 'bundle_metadata.json'
sample_ids_path = BUNDLE_ROOT / 'sample_ids.txt'
rows = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
bundle_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
if len(rows) != EXPECTED_BENCHMARK_EXAMPLES or len({row['sample_id'] for row in rows}) != len(rows):
    raise RuntimeError(f'expected exactly {EXPECTED_BENCHMARK_EXAMPLES} unique benchmark rows, got {len(rows)}')
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024): digest.update(chunk)
    return digest.hexdigest()
if not sample_ids_path.is_file(): raise FileNotFoundError(sample_ids_path)
sample_ids_text = sample_ids_path.read_text(encoding='utf-8')
if hashlib.sha256(sample_ids_text.encode('utf-8')).hexdigest() != bundle_metadata['sample_ids_sha256']:
    raise RuntimeError('sample_ids.txt hash does not match bundle metadata')
if sha256_file(manifest_path) != bundle_metadata['manifest_sha256']:
    raise RuntimeError('benchmark manifest hash does not match bundle metadata')
for row in rows:
    audio = BUNDLE_ROOT / row['audio_path']
    if not audio.is_file(): raise FileNotFoundError(audio)
    if sha256_file(audio) != row['audio_sha256']:
        raise RuntimeError(f'audio hash mismatch for {row["sample_id"]}')
print('samples=', len(rows))
print('sample_ids=', [row['sample_id'] for row in rows])
PERSIST_BENCHMARK_ROOT.mkdir(parents=True, exist_ok=True)
for source_path in (BUNDLE_ARCHIVE, manifest_path, sample_ids_path, metadata_path):
    shutil.copy2(source_path, PERSIST_BENCHMARK_ROOT / source_path.name)
print('bundle_metadata=', bundle_metadata)
print('persistent_benchmark_root=', PERSIST_BENCHMARK_ROOT)
print('PHASE_0_FILES_PASS')

## BLOCK 4 — Phase 0 stop gate

Confirm `PHASE_0_RUNTIME_PASS` and `PHASE_0_FILES_PASS`, then inspect the actual GPU, free VRAM, selected IDs, and audio hashes. The next cell installs only this notebook's dependencies.

In [ ]:
# BLOCK 5 — POST-APPROVAL ONLY — Teacher B's isolated dependencies.
%pip install -q 'transformers==5.16.1' accelerate sentencepiece soundfile scipy psutil huggingface_hub sacrebleu

In [ ]:
# BLOCK 6 — shared Teacher B utilities.
import soundfile as sf
from scipy.signal import resample_poly
TARGET_RATE = 16_000
def normalize_hindi(text: str) -> str:
    return ' '.join(unicodedata.normalize('NFC', text).split())
def read_audio(path: Path) -> tuple[np.ndarray, float, int]:
    array, sample_rate = sf.read(path, dtype='float32', always_2d=True)
    waveform = array.mean(axis=1) if array.shape[1] > 1 else array[:, 0]
    if sample_rate != TARGET_RATE:
        divisor = math.gcd(int(sample_rate), TARGET_RATE)
        waveform = resample_poly(waveform, TARGET_RATE // divisor, int(sample_rate) // divisor).astype('float32')
        sample_rate = TARGET_RATE
    duration = len(waveform) / TARGET_RATE
    if duration > MAX_AUDIO_SECONDS: raise ValueError(f'audio exceeds {MAX_AUDIO_SECONDS}s: {duration:.3f}s')
    return np.clip(waveform, -1.0, 1.0).astype('float32'), duration, int(sample_rate)
def existing_records(path: Path) -> dict[str, dict[str, object]]:
    if not path.is_file(): return {}
    return {str(row['sample_id']): row for row in (json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip())}
def append_jsonl(path: Path, row: dict[str, object]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n'); stream.flush()
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
durations = [read_audio(BUNDLE_ROOT / row['audio_path'])[1] for row in rows]
print('transformers=', __import__('transformers').__version__)
print('duration_min_median_p90_p95_max=', min(durations), statistics.median(durations), np.percentile(durations, 90), np.percentile(durations, 95), max(durations))

In [ ]:
# BLOCK 7 — POST-APPROVAL ONLY — verify the immutable smoke-tested model revision.
from huggingface_hub import HfApi
HF_TOKEN = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get('HF_TOKEN')
except Exception:
    pass
MODEL_REVISION = '5f8cc790b19fc3f67a61c105133b20b34e3dcb76'
resolved_revision = HfApi(token=HF_TOKEN).model_info(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN).sha
if resolved_revision != MODEL_REVISION:
    raise RuntimeError(f'model revision verification failed: expected {MODEL_REVISION}, got {resolved_revision}')
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'HF_TOKEN_available': bool(HF_TOKEN)})

In [ ]:
# BLOCK 8 — POST-APPROVAL ONLY — batch-1 deterministic speech-to-Hindi-text generation.
from transformers import AutoProcessor, SeamlessM4Tv2Model

output_path = OUTPUT_ROOT / 'teacher_seamless_m4t_v2_100.jsonl'
records = existing_records(output_path)
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
generation_config = {'src_lang': 'eng', 'tgt_lang': 'hin', 'generate_speech': False, 'text_num_beams': 4, 'text_do_sample': False, 'max_new_tokens': MAX_TRANSLATION_TOKENS, 'batch_size': 1}
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN)
model = SeamlessM4Tv2Model.from_pretrained(MODEL_ID, revision=MODEL_REVISION, torch_dtype=dtype, low_cpu_mem_usage=True, device_map='auto', token=HF_TOKEN)
model.eval()
def generated_ids(value):
    sequences = getattr(value, 'sequences', None)
    if sequences is not None: value = sequences
    elif isinstance(value, (tuple, list)): value = value[0]
    if value is None: raise RuntimeError('Seamless generation returned no token sequences')
    if not isinstance(value, torch.Tensor): value = torch.as_tensor(value)
    value = value.detach().cpu()
    if value.ndim == 1: return value.tolist()
    if value.ndim == 2: return value[0].tolist()
    raise RuntimeError(f'unexpected Seamless token shape: {tuple(value.shape)}')
model_device = model.device
model_dtype = next(parameter for parameter in model.parameters() if parameter.is_floating_point()).dtype
for row in rows:
    sample_id = row['sample_id']
    if sample_id in records and (records[sample_id].get('status') == 'ok' or not RETRY_FAILED): continue
    base = {'schema_version': 1, 'sample_id': sample_id, 'audio_path': row['audio_path'], 'audio_sha256': row['audio_sha256'], 'source_manifest_sha256': row.get('source_manifest_sha256'), 'source_dataset': row.get('source_dataset'), 'source_dataset_revision': row.get('source_dataset_revision'), 'source_license': row.get('source_license'), 'source_url': row.get('source_url'), 'source_split': row.get('source_split'), 'reference_hindi': row.get('reference_hindi'), 'source_text': row.get('source_text'), 'source_asr_text': row.get('source_asr_text'), 'teacher_pipeline': MODEL_ID, 'speech_teacher': MODEL_ID, 'speech_teacher_revision': MODEL_REVISION, 'speech_teacher_license': 'CC-BY-NC-4.0', 'translation_teacher': None, 'status': 'error', 'stage': 'seamless', 'generation_config': generation_config}
    started = time.perf_counter()
    try:
        waveform, duration, _ = read_audio(BUNDLE_ROOT / row['audio_path'])
        inputs = processor(audio=waveform, sampling_rate=TARGET_RATE, return_tensors='pt')
        inputs = {key: (value.to(device=model_device, dtype=model_dtype) if torch.is_floating_point(value) else value.to(model_device)) for key, value in inputs.items()}
        with torch.inference_mode():
            generated = model.generate(**inputs, tgt_lang='hin', generate_speech=False, text_num_beams=4, text_do_sample=False, max_new_tokens=MAX_TRANSLATION_TOKENS)
        raw_text = processor.decode(generated_ids(generated), skip_special_tokens=True)
        base.update({'status': 'ok', 'stage': None, 'teacher_hindi_model_decoded_raw': raw_text, 'teacher_hindi_raw': raw_text, 'teacher_hindi_normalized': normalize_hindi(raw_text), 'translation_seconds': time.perf_counter() - started, 'duration_seconds': duration})
    except torch.cuda.OutOfMemoryError as exc:
        clear_cuda(); base.update({'error_type': 'cuda_oom', 'error': repr(exc), 'translation_seconds': time.perf_counter() - started})
    except Exception as exc:
        base.update({'error_type': type(exc).__name__, 'error': repr(exc), 'translation_seconds': time.perf_counter() - started})
    append_jsonl(output_path, base); records[sample_id] = base
    print(sample_id, base['status'], base.get('error_type', 'ok'), base.get('error', ''))
del model, processor
clear_cuda()
print('canonical_output=', output_path)
print('translation_records=', len(records), 'successful=', sum(row.get('status') == 'ok' for row in records.values()), 'failed=', sum(row.get('status') != 'ok' for row in records.values()))

In [ ]:
# BLOCK 9 — Hindi-2K tokenizer audit. Upload the exact baseline tokenizer first.
import sentencepiece as spm
TOKENIZER_PATH = Path('/content/hindi_2k.model')
EXPECTED_TOKENIZER_SHA256 = '34b1c9c8f24b9fdb27a393978a0034fd96ea35b835b1a667e4373098a483137d'
MAX_SEMANTIC_TOKENS = 128
if not TOKENIZER_PATH.is_file(): raise FileNotFoundError(f'upload the exact baseline tokenizer first: {TOKENIZER_PATH}')
tokenizer_sha256 = sha256_file(TOKENIZER_PATH)
if tokenizer_sha256 != EXPECTED_TOKENIZER_SHA256: raise RuntimeError(f'tokenizer SHA-256 mismatch: expected {EXPECTED_TOKENIZER_SHA256}, got {tokenizer_sha256}')
tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER_PATH))
output_rows = [records[row['sample_id']] for row in rows if row['sample_id'] in records]
successful_outputs = [row for row in output_rows if row.get('status') == 'ok']
lengths = []; tokens_per_word = []; tokens_per_character = []; unknown = 0; total = 0; fragmentation_examples = []
for row in successful_outputs:
    text = str(row.get('teacher_hindi_raw', '')); ids = tokenizer.encode(text, out_type=int); pieces = tokenizer.encode(text, out_type=str)
    word_count = len(text.split()); character_count = sum(not character.isspace() for character in text)
    lengths.append(len(ids)); tokens_per_word.append(len(ids) / max(1, word_count)); tokens_per_character.append(len(ids) / max(1, character_count)); unknown += sum(token == tokenizer.unk_id() for token in ids); total += len(ids)
    if len(fragmentation_examples) < 20 and (any(character.isascii() and character.isalpha() for character in text) or any(character.isdigit() for character in text) or any(not character.isalnum() and not character.isspace() for character in text)):
        fragmentation_examples.append({'sample_id': row['sample_id'], 'teacher_hindi_raw': text, 'pieces': pieces, 'token_count': len(ids)})
def percentile(values, value): return float(np.percentile(values, value)) if values else None
tokenizer_audit = {'schema_version': 1, 'tokenizer_path': 'artifacts/tokenizers/hindi_2k/hindi.model', 'tokenizer_sha256': tokenizer_sha256, 'vocab_size': tokenizer.vocab_size(), 'max_semantic_tokens': MAX_SEMANTIC_TOKENS, 'successful_outputs': len(successful_outputs), 'unknown_token_count': unknown, 'unknown_token_rate': unknown / max(1, total), 'token_length_p50': percentile(lengths, 50), 'token_length_p90': percentile(lengths, 90), 'token_length_p95': percentile(lengths, 95), 'token_length_p99': percentile(lengths, 99), 'token_length_max': max(lengths) if lengths else None, 'mean_tokens_per_hindi_word': float(np.mean(tokens_per_word)) if tokens_per_word else None, 'mean_tokens_per_unicode_character': float(np.mean(tokens_per_character)) if tokens_per_character else None, 'percentage_exceeding_max_semantic_tokens': 100.0 * sum(length + 1 > MAX_SEMANTIC_TOKENS for length in lengths) / max(1, len(lengths)), 'fragmentation_examples': fragmentation_examples, 'lengths_include_special_tokens': False, 'exceedance_rule': 'raw SentencePiece pieces + BOS must fit within max_semantic_tokens'}
(OUTPUT_ROOT / 'teacher_b_tokenizer_audit.json').write_text(json.dumps(tokenizer_audit, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(tokenizer_audit, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
# BLOCK 10 — persist reproducibility metadata without secrets.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
metadata = {'schema_version': 1, 'teacher_pipeline': MODEL_ID, 'model_id': MODEL_ID, 'model_revision': globals().get('MODEL_REVISION'), 'model_license': 'CC-BY-NC-4.0', 'torch': torch.__version__, 'cuda': torch.version.cuda, 'transformers': __import__('transformers').__version__, 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', 'precision': str(dtype), 'bundle_metadata': bundle_metadata, 'generation_config': globals().get('generation_config', {}), 'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'benchmark_expected_examples': EXPECTED_BENCHMARK_EXAMPLES, 'persistent_root': str(PERSIST_ROOT), 'artifact_paths': {'teacher_b_jsonl': str(OUTPUT_ROOT / 'teacher_seamless_m4t_v2_100.jsonl'), 'tokenizer_audit': str(OUTPUT_ROOT / 'teacher_b_tokenizer_audit.json')}}
(OUTPUT_ROOT / 'teacher_b_experiment_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(metadata, indent=2, ensure_ascii=False, sort_keys=True))
print('teacher_b_metadata=', OUTPUT_ROOT / 'teacher_b_experiment_metadata.json')

In [ ]:
# BLOCK 11 — Teacher B metrics, failures, and manual examples.
import sacrebleu
def _norm(text): return ' '.join(unicodedata.normalize('NFC', str(text or '')).split())
def _diagnostics(text, source):
    text = _norm(text); source = _norm(source); words = text.split(); source_words = source.split(); devanagari = sum('\u0900' <= char <= '\u097f' for char in text); letters = sum(char.isalpha() for char in text); latin_ratio = sum('a' <= char.lower() <= 'z' for char in text) / max(1, letters); repeated = sum(a == b for a, b in zip(words, words[1:]))
    return {'empty': not text, 'mostly_latin': bool(letters) and latin_ratio > 0.8, 'devanagari_ratio': devanagari / max(1, len(text)), 'adjacent_word_repetition_rate': repeated / max(1, len(words) - 1), 'source_copy': bool(text) and text.casefold() == source.casefold(), 'word_count': len(words), 'source_word_count': len(source_words), 'target_source_word_ratio': len(words) / max(1, len(source_words))}
output_rows = [records.get(row['sample_id'], {'sample_id': row['sample_id'], 'status': 'missing'}) for row in rows]
successful = [row for row in output_rows if row.get('status') == 'ok']
hypotheses = [_norm(row.get('teacher_hindi_raw', '')) for row in successful]; references = [_norm(row.get('reference_hindi', '')) for row in successful]; diagnostics = [_diagnostics(row.get('teacher_hindi_raw', ''), row.get('source_text', '')) for row in successful]
ratios = [float(item['target_source_word_ratio']) for item in diagnostics]
teacher_b_metrics = {'schema_version': 1, 'requested_samples': len(rows), 'successful_samples': len(successful), 'failed_samples': len(rows) - len(successful), 'bleu': float(sacrebleu.corpus_bleu(hypotheses, [references], tokenize='intl', use_effective_order=True).score) if successful else None, 'chrf2': float(sacrebleu.corpus_chrf(hypotheses, [references], word_order=2).score) if successful else None, 'exact_match_count': sum(h == r for h, r in zip(hypotheses, references, strict=True)), 'empty_output_count': sum(item['empty'] for item in diagnostics), 'mostly_latin_count': sum(item['mostly_latin'] for item in diagnostics), 'source_copy_count': sum(item['source_copy'] for item in diagnostics), 'mean_devanagari_ratio': float(np.mean([item['devanagari_ratio'] for item in diagnostics])) if diagnostics else None, 'mean_adjacent_word_repetition_rate': float(np.mean([item['adjacent_word_repetition_rate'] for item in diagnostics])) if diagnostics else None, 'target_word_count_p50_p90_p95_p99_max': [float(np.percentile([item['word_count'] for item in diagnostics], value)) for value in (50, 90, 95, 99)] + ([max(item['word_count'] for item in diagnostics)] if diagnostics else [None]), 'target_source_word_ratio_p50_p90_p95_p99_max': [float(np.percentile(ratios, value)) for value in (50, 90, 95, 99)] + ([max(ratios)] if ratios else [None]), 'translation_total_seconds': sum(float(row.get('translation_seconds', 0.0) or 0.0) for row in records.values()), 'translation_average_seconds': sum(float(row.get('translation_seconds', 0.0) or 0.0) for row in records.values()) / max(1, len(rows))}
failure_rows = [row for row in output_rows if row.get('status') != 'ok']
with (OUTPUT_ROOT / 'teacher_b_failures.jsonl').open('w', encoding='utf-8') as stream:
    for row in failure_rows: stream.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + '\n')
ranked = []
for row, diagnostic in zip(successful, diagnostics, strict=True):
    score = (5 if diagnostic['empty'] else 0) + (4 if diagnostic['mostly_latin'] else 0) + (3 if diagnostic['source_copy'] else 0) + (3 if diagnostic['adjacent_word_repetition_rate'] > 0 else 0) + (2 if any(character.isdigit() for character in str(row.get('teacher_hindi_raw', ''))) else 0)
    ranked.append((score, row['sample_id']))
manual_ids = [row['sample_id'] for row in failure_rows]
for _, sample_id in sorted(ranked, key=lambda item: (-item[0], item[1])):
    if sample_id not in manual_ids: manual_ids.append(sample_id)
    if len(manual_ids) >= 20: break
manual_ids = manual_ids[:20]; benchmark_by_id = {row['sample_id']: row for row in rows}; manual_examples = []
for sample_id in manual_ids:
    bundle_row = benchmark_by_id[sample_id]; output_row = records.get(sample_id, {})
    manual_examples.append({'sample_id': sample_id, 'reference_english': bundle_row.get('source_asr_text') or bundle_row.get('source_text'), 'human_hindi': bundle_row.get('reference_hindi'), 'teacher_b_hindi': output_row.get('teacher_hindi_raw'), 'status': output_row.get('status', 'missing'), 'error': output_row.get('error')})
(OUTPUT_ROOT / 'teacher_b_metrics.json').write_text(json.dumps(teacher_b_metrics, ensure_ascii=False, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(OUTPUT_ROOT / 'teacher_b_manual_examples.json').write_text(json.dumps(manual_examples, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
with (OUTPUT_ROOT / 'teacher_b_manual_examples.md').open('w', encoding='utf-8') as stream:
    for example in manual_examples: stream.write(f"## {example['sample_id']}\n\nReference English: {example.get('reference_english', '')}\n\nHuman Hindi: {example.get('human_hindi', '')}\n\nTeacher B Hindi: {example.get('teacher_b_hindi', '')}\n\nStatus: {example.get('status')}\n\n---\n\n")
print(json.dumps(teacher_b_metrics, ensure_ascii=False, indent=2, sort_keys=True))
print('teacher_b_manual_examples=', OUTPUT_ROOT / 'teacher_b_manual_examples.md')

In [ ]:
# BLOCK 12 — verify and package persistent Teacher B artifacts.
import zipfile
required_artifacts = [PERSIST_BENCHMARK_ROOT / 'benchmark_manifest.jsonl', PERSIST_BENCHMARK_ROOT / 'sample_ids.txt', PERSIST_BENCHMARK_ROOT / 'bundle_metadata.json', PERSIST_BENCHMARK_ROOT / BUNDLE_ARCHIVE.name, OUTPUT_ROOT / 'teacher_seamless_m4t_v2_100.jsonl', OUTPUT_ROOT / 'teacher_b_experiment_metadata.json', OUTPUT_ROOT / 'teacher_b_metrics.json', OUTPUT_ROOT / 'teacher_b_tokenizer_audit.json', OUTPUT_ROOT / 'teacher_b_manual_examples.json', OUTPUT_ROOT / 'teacher_b_manual_examples.md', OUTPUT_ROOT / 'teacher_b_failures.jsonl']
missing = [str(path) for path in required_artifacts if not path.is_file()]
if missing: raise FileNotFoundError('missing persistent artifacts: ' + ', '.join(missing))
archive_path = PERSIST_ROOT / 'teacher_b_100_artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in required_artifacts: archive.write(path, arcname=path.relative_to(PERSIST_ROOT))
print('teacher_b_archive=', archive_path)
print('teacher_b_archive_bytes=', archive_path.stat().st_size)
print('TEACHER_B_STOP_GATE_READY')